In [ ]:
# Cell 1: Execution Timer, Safety Limits & Dependency Setup
import os
import sys
import time

START_TIME = time.time()
MAX_HOURS = 11.75  # 11h 45m safety cutoff
MAX_SECONDS = MAX_HOURS * 3600

print(f"=== Execution started at: {time.ctime(START_TIME)} ===")
print(f"Safety deadline: {MAX_HOURS:.2f} hours ({MAX_SECONDS:.0f} seconds)")

def time_elapsed():
    return time.time() - START_TIME

def time_remaining():
    return max(0.0, MAX_SECONDS - time_elapsed())

def check_deadline(stage_name="checkpoint"):
    elapsed = time_elapsed()
    rem = time_remaining()
    print(f"[{stage_name}] Elapsed: {elapsed/3600:.2f}h | Remaining: {rem/3600:.2f}h")
    if elapsed >= MAX_SECONDS:
        print(f"CRITICAL: 11h45m safety limit reached ({elapsed/3600:.2f}h). Triggering emergency save & halt.")
        return True
    return False

# Uninstall incompatible torchao and install required dependencies cleanly
!pip uninstall -y torchao
!pip install -q -U "transformers>=4.48.0" trl peft accelerate bitsandbytes datasets sentencepiece protobuf gguf


In [ ]:
# Cell 2: M4 Tool Definitions matching Micro-Model Native Harness
M4_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_repo_map",
            "description": "Return the skeleton of a file or directory: every class and function with its signature and docstring, with all bodies hidden.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": [],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_symbol",
            "description": "Load the full source of exactly one function or class, by name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file": {"type": "string"},
                    "symbol_name": {"type": "string"},
                },
                "required": ["file", "symbol_name"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_text",
            "description": "Substring grep across the repository with file, line numbers, and context lines.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pattern": {"type": "string"},
                    "path": {"type": "string"},
                },
                "required": ["pattern"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read file contents around a focus region, optionally with line numbers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string"},
                    "start_line": {"type": "integer"},
                    "end_line": {"type": "integer"},
                },
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_edit",
            "description": "Replace exactly one occurrences of search_block with replace_block inside target_symbol in file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file": {"type": "string"},
                    "target_symbol": {"type": "string"},
                    "search_block": {"type": "string"},
                    "replace_block": {"type": "string"},
                },
                "required": ["file", "target_symbol", "search_block", "replace_block"],
                "additionalProperties": False,
            },
        },
    },
]
print(f"Registered {len(M4_TOOLS)} M4 tool schemas.")


In [ ]:
# Cell 3: Load Gemma 4 E4B & Configure QLoRA (Freezing PLE)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import gc
import glob
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# Free any stray allocations
gc.collect()
torch.cuda.empty_cache()

# Scan /kaggle/input for mounted model
MODEL_PATH = "google/gemma-4-E4B-it"
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files and any(x in root.lower() for x in ["gemma", "e4b"]):
        MODEL_PATH = root
        print(f"Found mounted model at: {MODEL_PATH}")
        break

print(f"Loading model from: {MODEL_PATH}")

MAX_SEQ_LENGTH = 1536

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map={"": 0},
    attn_implementation="eager",
)

# Freeze base model parameters without bulk fp32 cast
for param in model.parameters():
    param.requires_grad = False

# Target ONLY the language model attention and MLP layers (bypassing vision & audio towers)
target_modules = []
for name, module in model.named_modules():
    if "language_model" in name:
        if any(name.endswith(f".{proj}") for proj in ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]):
            target_modules.append(name)

print(f"Targeting {len(target_modules)} language model linear projections (first 4: {target_modules[:4]})")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)
model = get_peft_model(model, lora_config)

# Precise dtype alignment for Nvidia T4 CC 7.5:
# 1. Base model frozen parameters & buffers -> torch.float16 (eliminates BFloat16 GEMM error)
for name, param in model.named_parameters():
    if not param.requires_grad and param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
for buf in model.buffers():
    if buf.dtype == torch.bfloat16:
        buf.data = buf.data.to(torch.float16)

# 2. LoRA trainable parameters -> strictly torch.float32
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

trainable_dtypes = {p.dtype for p in model.parameters() if p.requires_grad}
frozen_dtypes = {p.dtype for p in model.parameters() if not p.requires_grad}
print(f"Trainable dtypes: {trainable_dtypes}")
print(f"Frozen dtypes: {frozen_dtypes} (no BFloat16 allowed on T4)")

model.print_trainable_parameters()
check_deadline("Model loaded")


In [ ]:
# Cell 4: Load & Tokenize daoviet/gemma4-m4-sft-3000
import json
from datasets import Dataset

DATASET_PATH = "/kaggle/input/gemma4-m4-sft-3000/sft_m4_3000.jsonl"
for root, dirs, files in os.walk("/kaggle/input"):
    if "sft_m4_3000.jsonl" in files:
        DATASET_PATH = os.path.join(root, "sft_m4_3000.jsonl")
        print(f"Found dataset at: {DATASET_PATH}")
        break

print(f"Reading dataset from: {DATASET_PATH}")
raw_rows = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_rows.append(json.loads(line))

print(f"Total rows in file: {len(raw_rows)}")

# Filter to avoid peak logits OOM on 16GB GPU
raw_rows = [r for r in raw_rows if sum(len(m.get("content", "") or "") for m in r.get("messages", [])) <= 6000]
print(f"Rows retained after length filtering: {len(raw_rows)}")

def format_row(example):
    msgs = example["messages"]
    try:
        text = tokenizer.apply_chat_template(
            msgs,
            tools=M4_TOOLS,
            tokenize=False,
            add_generation_prompt=False,
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
    return {"text": text}

ds = Dataset.from_list([format_row(r) for r in raw_rows]).train_test_split(test_size=0.05, seed=42)
print("Rendered example tail:\n", ds["train"][0]["text"][-350:])
check_deadline("Dataset processed")


In [ ]:
# Cell 5: Time Tracking & Early Stop Callback
from transformers import TrainerCallback

def save_checkpoint(tag="final"):
    out_dir = f"/kaggle/working/gemma-4-E4B-M4-lora-{tag}"
    print(f"Saving LoRA checkpoint to: {out_dir}")
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    print("Checkpoint saved successfully.")

class TimeLimitCallback(TrainerCallback):
    def __init__(self, max_seconds, save_fn):
        self.max_seconds = max_seconds
        self.save_fn = save_fn
        self.saved = False

    def on_step_end(self, args, state, control, **kwargs):
        elapsed = time_elapsed()
        if elapsed >= self.max_seconds:
            print(f"\n[ALERT] Reached safety cutoff at step {state.global_step} ({elapsed/3600:.2f}h).")
            print("Requesting training halt and executing emergency checkpoint save...")
            control.should_training_stop = True
            if not self.saved:
                self.save_fn("emergency_11h45m_cutoff")
                self.saved = True


In [ ]:
# Cell 6: Train with SFTTrainer
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "/kaggle/working/gemma-4-E4B-M4-lora-run"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sanity check gradient flow on a mini batch before training
print("Running single-sample gradient flow verification...")
sample_text = ds["train"][0]["text"][:400]
test_tok = tokenizer(sample_text, return_tensors="pt").to(model.device)
test_tok["labels"] = test_tok["input_ids"]
test_out = model(**test_tok)
test_out.loss.backward()
grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
print(f"Gradient verification passed: {len(grads)} trainable tensors have valid gradients (loss={test_out.loss.item():.4f})")
model.zero_grad()

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1.0e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    num_train_epochs=2,
    fp16=False,
    bf16=False,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    optim="adamw_torch",
    gradient_checkpointing=True,
    report_to="none",
)

time_callback = TimeLimitCallback(max_seconds=MAX_SECONDS, save_fn=save_checkpoint)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    peft_config=None,  # model is already a PeftModel
    processing_class=tokenizer,
    callbacks=[time_callback],
    args=sft_config,
)

print("Starting training loop...")
trainer.train()

save_checkpoint("completed")
check_deadline("Training completed")


In [ ]:
# Cell 7: Sanity Check Generation on Held-Out Validation Task
print("=== Running Sanity Check Generation ===")
val_sample = raw_rows[-1]["messages"][:2]
try:
    val_prompt = tokenizer.apply_chat_template(
        val_sample,
        tools=M4_TOOLS,
        tokenize=False,
        add_generation_prompt=True,
    )
except TypeError:
    val_prompt = tokenizer.apply_chat_template(
        val_sample,
        tokenize=False,
        add_generation_prompt=True,
    )

inputs = tokenizer(val_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
gen_text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
print("Generated Tool Call Output:\n", gen_text)


In [ ]:
# Cell 8: Merge LoRA Weights & Export
if not check_deadline("Pre-export"):
    print("=== Merging LoRA Weights into Base Model ===")
    
    # 1. Clear GPU memory
    del model
    del trainer
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    # 2. Free intermediate training checkpoints to preserve Kaggle 20GB disk limit
    import shutil
    shutil.rmtree("/kaggle/working/gemma-4-E4B-M4-lora-run", ignore_errors=True)
    !rm -rf /root/.cache /tmp/*

    # 3. Locate latest LoRA
    latest_lora = "/kaggle/working/gemma-4-E4B-M4-lora-completed"
    if not os.path.exists(latest_lora):
        candidates = sorted(glob.glob("/kaggle/working/gemma-4-E4B-M4-lora-*"))
        if candidates:
            latest_lora = candidates[-1]

    print(f"Using LoRA adapter from: {latest_lora}")

    # Ensure PEFT does not invoke incompatible torchao dispatchers
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False

    from peft import PeftModel
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16,
        device_map="cpu",
        low_cpu_mem_usage=True,
        attn_implementation="eager",
    )
    merged_model = PeftModel.from_pretrained(base_model, latest_lora)
    merged_model = merged_model.merge_and_unload()

    MERGED_DIR = "/kaggle/working/gemma-4-E4B-M4-sft-fp16"
    print(f"Saving merged FP16 model to: {MERGED_DIR}")
    merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)
    del base_model
    del merged_model
    gc.collect()
    print("Merged FP16 model saved successfully.")

    # 4. Export LoRA adapter to GGUF format for llama.cpp (~70 MB, highly disk-efficient)
    print("Attempting LoRA GGUF export via llama.cpp...")
    import subprocess
    try:
        subprocess.run(["pip", "install", "-q", "gguf"], check=False)
        if not os.path.exists("/tmp/llama.cpp"):
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", "/tmp/llama.cpp"], check=True)
        env = os.environ.copy()
        env["PYTHONPATH"] = "/tmp/llama.cpp:" + env.get("PYTHONPATH", "")
        lora_gguf_out = "/kaggle/working/gemma-4-E4B-M4-sft-lora.gguf"
        res = subprocess.run(
            ["python", "/tmp/llama.cpp/convert_lora_to_gguf.py", latest_lora, "--base", MODEL_PATH, "--outfile", lora_gguf_out],
            cwd="/tmp/llama.cpp",
            capture_output=True,
            text=True,
            env=env,
        )
        print("llama.cpp convert_lora stdout:\n", res.stdout[-1000:] if res.stdout else "None")
        if res.returncode == 0 and os.path.exists(lora_gguf_out):
            print(f"=== LoRA GGUF Export SUCCESS! Size: {os.path.getsize(lora_gguf_out)/(1024*1024):.2f} MB ===")
        else:
            print("llama.cpp convert_lora stderr:\n", res.stderr[-1000:] if res.stderr else "None")
            print("Notice: Full merged FP16 model and LoRA checkpoints are preserved.")
    except Exception as e:
        print(f"LoRA GGUF step encountered notice: {e}")

    # Check remaining disk space
    free_gb = shutil.disk_usage("/kaggle/working").free / (1024**3)
    print(f"Safe disk headroom remaining: {free_gb:.2f} GB")

    print("=== Final working directory contents ===")
    for item in os.listdir("/kaggle/working"):
        item_path = os.path.join("/kaggle/working", item)
        if os.path.isdir(item_path):
            print(f"  [DIR]  {item}")
        else:
            print(f"  [FILE] {item} ({os.path.getsize(item_path)/(1024*1024):.2f} MB)")
